# NAIP ZIP Organizer: Prefire/Postfire + Unzip

This notebook organizes NAIP imagery ZIP files into `prefire` and `postfire` folders based on the acquisition date encoded in each ZIP filename, then unzips each file into a folder named after the ZIP filename stem.

## What this does

1. Takes a list of folder paths that contain NAIP ZIP files.
2. Parses acquisition date from filenames like `m_3611835_se_11_060_20220707.ZIP` (last underscore token = `YYYYMMDD`).
3. Classifies files as `prefire` or `postfire` from a fire year.
4. Moves ZIP files into `prefire` / `postfire` subfolders.
5. Unzips each ZIP into a same-name folder in its destination folder.

By default, `SAME_YEAR_IS_PREFIRE = True`, so files acquired during or before the configured fire year are `prefire`; later files are `postfire`.

In [ ]:
from pathlib import Path
import shutil
import subprocess
from datetime import datetime, date

try:
    from osgeo import gdal
    gdal.UseExceptions()
except Exception:
    gdal = None

def first_existing_path(*candidates: str | Path) -> Path:
    # Return the first path that exists from a list of likely locations.
    # This makes the notebook work whether the kernel starts in the repo root
    # or in the notebooks/ folder.
    paths = [Path(candidate) for candidate in candidates]
    for path in paths:
        if path.exists():
            return path
    return paths[0]


# Folders that contain NAIP ZIP files (edit this list).
NAIP_FOLDERS = [
    first_existing_path('./downloads/czu_aug_lightning_2020/naip', './notebooks/downloads/czu_aug_lightning_2020/naip'),
    # first_existing_path('./downloads/creek/naip', './notebooks/downloads/creek/naip'),
    # first_existing_path('./downloads/czu/naip', './notebooks/downloads/czu/naip'),
    # first_existing_path('./downloads/northcomplex/naip', './notebooks/downloads/northcomplex/naip'),
]

# Fire year mapping (keyed by folder path string).
# You can define per-folder fire years or use DEFAULT_FIRE_YEAR.
FIRE_YEAR_BY_FOLDER = {
    './downloads/czu_aug_lightning_2020/naip': 2020,
    './notebooks/downloads/czu_aug_lightning_2020/naip': 2020,
    # './downloads/creek/naip': 2020,
    # './notebooks/downloads/creek/naip': 2020,
}
DEFAULT_FIRE_YEAR = 2020  # Example: 2020

# Optional per-folder exact fire date (YYYY-MM-DD).
# If provided, this takes precedence over year-based classification.
FIRE_DATE_BY_FOLDER = {
    # './downloads/castle/naip': '2020-08-16',
}

# Year-based rule fallback when exact fire date is not provided.
# True means imagery from the fire year is treated as prefire.
SAME_YEAR_IS_PREFIRE = True

# Safety limits for sanity checks before move/unzip.
MAX_MOVE_FILES = 5000
MAX_UNZIP_FILES = 5000
MAX_ESTIMATED_UNZIP_GB = 200.0

# If True, print planned operations but do not move, purge, or unzip.
DRY_RUN = False

# Invalid ZIPs are moved here before unzipping so they cannot block valid files.
INVALID_ZIP_PURGE_DIRNAME = '_invalid_zips_for_purge'

print('Configured NAIP folders:')
for folder in NAIP_FOLDERS:
    prefire_count = sum(1 for path in (folder / 'prefire').rglob('*') if path.is_file() and path.suffix.lower() == '.zip') if (folder / 'prefire').exists() else 0
    postfire_count = sum(1 for path in (folder / 'postfire').rglob('*') if path.is_file() and path.suffix.lower() == '.zip') if (folder / 'postfire').exists() else 0
    print(f'  {folder.resolve()}')
    print(f'    exists={folder.exists()} prefire_zips={prefire_count} postfire_zips={postfire_count}')

In [ ]:
def parse_acquisition_date_from_name(zip_path: Path) -> datetime:
    """Parse acquisition date from final underscore token (YYYYMMDD)."""
    stem = zip_path.stem
    date_token = stem.rsplit('_', 1)[-1]
    return datetime.strptime(date_token, '%Y%m%d')


def resolve_fire_year(folder: Path) -> int:
    folder_key = str(folder)
    fire_year = FIRE_YEAR_BY_FOLDER.get(folder_key, DEFAULT_FIRE_YEAR)
    if fire_year is None:
        raise ValueError(
            f'No fire year configured for {folder_key}. Add it to FIRE_YEAR_BY_FOLDER or set DEFAULT_FIRE_YEAR.'
        )
    return int(fire_year)


def resolve_fire_date(folder: Path):
    folder_key = str(folder)
    fire_date_text = FIRE_DATE_BY_FOLDER.get(folder_key)
    if not fire_date_text:
        return None
    return datetime.strptime(fire_date_text, '%Y-%m-%d').date()


def classify_prefire_postfire(acq_date: datetime, fire_year: int, fire_date: date | None = None) -> str:
    acq_day = acq_date.date()
    if fire_date is not None:
        # Acquisition on or before fire date is treated as prefire.
        return 'prefire' if acq_day <= fire_date else 'postfire'

    if SAME_YEAR_IS_PREFIRE:
        return 'prefire' if acq_date.year <= fire_year else 'postfire'
    return 'prefire' if acq_date.year < fire_year else 'postfire'


def find_zip_files(folder: Path):
    # Recursive search for .zip/.ZIP while skipping already-organized output folders.
    for candidate in folder.rglob('*'):
        if not candidate.is_file():
            continue
        if candidate.suffix.lower() != '.zip':
            continue
        if {'prefire', 'postfire'}.intersection(set(candidate.parts)):
            continue
        yield candidate


def find_bucket_zip_files(folder: Path, bucket: str):
    bucket_dir = folder / bucket
    if not bucket_dir.exists():
        return
    for candidate in bucket_dir.rglob('*'):
        if candidate.is_file() and candidate.suffix.lower() == '.zip':
            yield candidate


def gather_zip_candidates_for_classification(folder: Path):
    # Include unorganized ZIPs and already bucketed ZIPs so reruns can fix misclassification.
    seen = set()
    for candidate in find_zip_files(folder):
        seen.add(candidate)
    for bucket in ['prefire', 'postfire']:
        for candidate in find_bucket_zip_files(folder, bucket):
            seen.add(candidate)
    return sorted(seen)


def human_size(num_bytes: int) -> str:
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f'{size:.2f} {unit}'
        size /= 1024


def zip_storage_stats(zip_path: Path) -> tuple[int, int]:
    """Return (compressed_bytes, estimated_uncompressed_bytes) using unzip."""
    compressed = zip_path.stat().st_size
    result = subprocess.run(
        ['unzip', '-l', str(zip_path)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())

    # The summary line usually looks like: "493082983  1 file".
    uncompressed = 0
    for line in reversed(result.stdout.splitlines()):
        parts = line.strip().split()
        if len(parts) >= 2 and parts[0].isdigit() and parts[1].startswith('file'):
            uncompressed = int(parts[0])
            break
        if len(parts) >= 3 and parts[0].isdigit() and parts[2].startswith('file'):
            uncompressed = int(parts[0])
            break
    if uncompressed == 0:
        raise RuntimeError(f'unzip did not report an uncompressed size for {zip_path}')
    return compressed, uncompressed


def validate_zip_with_unzip(zip_path: Path) -> tuple[bool, str]:
    """Validate a ZIP archive with unzip -t before extraction."""
    result = subprocess.run(
        ['unzip', '-t', str(zip_path)],
        capture_output=True,
        text=True,
    )
    message = (result.stderr.strip() or result.stdout.strip()).splitlines()
    detail = message[-1] if message else ''
    return result.returncode == 0, detail


def purge_path_for_zip(base_folder: Path, zip_path: Path) -> Path:
    """Return destination path for an invalid ZIP, preserving relative folders."""
    relative_zip = zip_path.relative_to(base_folder)
    return base_folder / INVALID_ZIP_PURGE_DIRNAME / relative_zip


def move_invalid_zip_to_purge(base_folder: Path, zip_path: Path) -> Path:
    """Move one invalid ZIP into the purge folder, avoiding name collisions."""
    dst = purge_path_for_zip(base_folder, zip_path)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        stem = dst.stem
        suffix = dst.suffix
        n = 1
        while True:
            candidate = dst.with_name(f'{stem}_{n}{suffix}')
            if not candidate.exists():
                dst = candidate
                break
            n += 1
    shutil.move(str(zip_path), str(dst))
    return dst

In [ ]:
move_plan = []
planning_skips = 0
already_in_correct_bucket = 0
total_move_zip_bytes = 0

for base_folder in NAIP_FOLDERS:
    base_folder = Path(base_folder)
    if not base_folder.exists():
        print(f'[skip] folder does not exist: {base_folder}')
        planning_skips += 1
        continue

    fire_year = resolve_fire_year(base_folder)
    fire_date = resolve_fire_date(base_folder)

    for zip_path in gather_zip_candidates_for_classification(base_folder):
        try:
            acq_date = parse_acquisition_date_from_name(zip_path)
        except Exception as exc:
            print(f'[skip] could not parse date from {zip_path.name}: {exc}')
            planning_skips += 1
            continue

        bucket = classify_prefire_postfire(acq_date, fire_year, fire_date)
        dest_dir = base_folder / bucket
        dest_zip = dest_dir / zip_path.name

        if zip_path == dest_zip:
            already_in_correct_bucket += 1
            continue

        move_plan.append((zip_path, dest_zip, acq_date.date(), fire_year, fire_date, bucket))
        total_move_zip_bytes += zip_path.stat().st_size

# Report plan details before moving anything.
prefire_count = sum(1 for *_, bucket in move_plan if bucket == 'prefire')
postfire_count = sum(1 for *_, bucket in move_plan if bucket == 'postfire')
print(f'Planned ZIP moves: {len(move_plan)}')
print(f'  prefire={prefire_count}, postfire={postfire_count}, skipped={planning_skips}')
print(f'  already_in_correct_bucket={already_in_correct_bucket}')
print(f'  compressed size to move: {human_size(total_move_zip_bytes)}')

for src, dst, acq_date, fire_year, fire_date, bucket in move_plan[:20]:
    fire_marker = fire_date.isoformat() if fire_date else f'year={fire_year}'
    print(f'  {src} -> {dst} | acq_date={acq_date}, fire={fire_marker}, bucket={bucket}')
if len(move_plan) > 20:
    print(f'  ... and {len(move_plan)-20} more')

# Safety checks before major file operations.
if not DRY_RUN and len(move_plan) > MAX_MOVE_FILES:
    raise RuntimeError(
        f'Planned move count ({len(move_plan)}) exceeds MAX_MOVE_FILES ({MAX_MOVE_FILES}).'
    )

if not DRY_RUN:
    moved = 0
    destination_exists = 0
    for src, dst, *_ in move_plan:
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            destination_exists += 1
            print(f'[skip] destination exists: {dst}')
            continue
        shutil.move(str(src), str(dst))
        moved += 1
        print(f'[move] {src} -> {dst}')
    print(f'Move step complete. moved={moved}, destination_exists={destination_exists}')
else:
    print('DRY_RUN=True, no files moved yet.')

In [ ]:
def folder_has_readable_tif(folder: Path) -> bool:
    """Return True when a folder contains at least one readable TIFF file."""
    if not folder.exists():
        return False
    tif_paths = [
        path for path in folder.rglob('*')
        if path.is_file() and path.suffix.lower() in {'.tif', '.tiff'}
    ]
    if not tif_paths:
        return False
    if gdal is None:
        return True
    for tif_path in tif_paths:
        try:
            ds = gdal.Open(str(tif_path), gdal.GA_ReadOnly)
            if ds is not None:
                ds = None
                return True
        except Exception:
            continue
    return False


def unzip_into_named_folder(zip_path: Path, overwrite: bool = False) -> Path:
    extract_dir = zip_path.with_suffix('')
    if folder_has_readable_tif(extract_dir) and not overwrite:
        return extract_dir
    if extract_dir.exists() and not folder_has_readable_tif(extract_dir):
        # Remove a partial/failed extraction from an earlier run before retrying.
        shutil.rmtree(extract_dir)

    work_dir = extract_dir.with_name(f'{extract_dir.name}_extracting')
    if work_dir.exists():
        shutil.rmtree(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ['unzip', '-q', str(zip_path.resolve()), '-d', str(work_dir.resolve())],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        shutil.rmtree(work_dir)
        raise RuntimeError(
            f'unzip could not extract {zip_path}: {(result.stderr.strip() or result.stdout.strip())}'
        )

    if not folder_has_readable_tif(work_dir):
        shutil.rmtree(work_dir)
        raise RuntimeError(f'No TIFF files were extracted from {zip_path}')

    if extract_dir.exists() and overwrite:
        shutil.rmtree(extract_dir)
    if not extract_dir.exists():
        work_dir.rename(extract_dir)
    else:
        for path in work_dir.iterdir():
            target = extract_dir / path.name
            if not target.exists():
                path.rename(target)
        shutil.rmtree(work_dir)
    return extract_dir


zip_targets = []
invalid_archives = []
total_compressed = 0
total_uncompressed_est = 0


# Preflight validity checks and storage impact estimates.
for base_folder in NAIP_FOLDERS:
    base_folder = Path(base_folder)
    for bucket in ['prefire', 'postfire']:
        for zip_path in find_bucket_zip_files(base_folder, bucket):
            is_valid, validation_message = validate_zip_with_unzip(zip_path)
            if not is_valid:
                invalid_archives.append((base_folder, zip_path, validation_message))
                continue

            try:
                compressed, uncompressed = zip_storage_stats(zip_path)
            except Exception as exc:
                print(f'[warn] unable to inspect {zip_path}: {exc}')
                invalid_archives.append((base_folder, zip_path, str(exc)))
                continue

            zip_targets.append(zip_path)
            total_compressed += compressed
            total_uncompressed_est += uncompressed

print(f'ZIPs available for unzip: {len(zip_targets)}')
print(f'  compressed size: {human_size(total_compressed)}')
print(f'  estimated uncompressed size: {human_size(total_uncompressed_est)}')
if total_compressed > 0:
    ratio = total_uncompressed_est / total_compressed
    print(f'  estimated expansion ratio: {ratio:.2f}x')

if invalid_archives:
    print(f'Invalid ZIP files: {len(invalid_archives)}')
    for _base_folder, path, reason in invalid_archives[:20]:
        print(f'  [invalid] {path}: {reason}')
    if len(invalid_archives) > 20:
        print(f'  ... and {len(invalid_archives)-20} more')

if invalid_archives and DRY_RUN:
    for base_folder, zip_path, _reason in invalid_archives[:20]:
        print(f'[dry-run purge] {zip_path} -> {purge_path_for_zip(base_folder, zip_path)}')
elif invalid_archives:
    if len(invalid_archives) > MAX_MOVE_FILES:
        raise RuntimeError(
            f'Invalid ZIP count ({len(invalid_archives)}) exceeds MAX_MOVE_FILES ({MAX_MOVE_FILES}).'
        )
    moved_invalid = 0
    for base_folder, zip_path, _reason in invalid_archives:
        if not zip_path.exists():
            continue
        purge_path = move_invalid_zip_to_purge(base_folder, zip_path)
        moved_invalid += 1
        print(f'[purge] {zip_path} -> {purge_path}')
    print(f'Moved invalid ZIPs to purge folder: {moved_invalid}')

if not DRY_RUN and len(zip_targets) > MAX_UNZIP_FILES:
    raise RuntimeError(
        f'Planned unzip count ({len(zip_targets)}) exceeds MAX_UNZIP_FILES ({MAX_UNZIP_FILES}).'
    )

if not DRY_RUN and (total_uncompressed_est / (1024 ** 3)) > MAX_ESTIMATED_UNZIP_GB:
    raise RuntimeError(
        'Estimated uncompressed size exceeds MAX_ESTIMATED_UNZIP_GB. '
        f'Estimate={(total_uncompressed_est / (1024 ** 3)):.2f} GB, limit={MAX_ESTIMATED_UNZIP_GB:.2f} GB.'
    )

unzipped = 0
already_present = 0
unzip_errors = []

for zip_path in zip_targets:
    extract_dir = zip_path.with_suffix('')
    if folder_has_readable_tif(extract_dir):
        already_present += 1
        print(f'[skip] already extracted: {extract_dir}')
        continue

    if DRY_RUN:
        print(f'[dry-run unzip] {zip_path} -> {extract_dir}')
        continue

    try:
        out_dir = unzip_into_named_folder(zip_path, overwrite=False)
        unzipped += 1
        print(f'[unzipped] {zip_path} -> {out_dir}')
    except Exception as exc:
        unzip_errors.append((zip_path, exc))
        print(f'[error unzip] {zip_path}: {exc}')

if DRY_RUN:
    print('DRY_RUN=True, no ZIPs were extracted.')
else:
    print(f'Completed unzip step. unzipped={unzipped}, already_present={already_present}, errors={len(unzip_errors)}')
    if unzip_errors:
        print('Archives that could not be fully extracted:')
        for path, exc in unzip_errors[:20]:
            print(f'  [failed] {path}: {exc}')
        if len(unzip_errors) > 20:
            print(f'  ... and {len(unzip_errors)-20} more')

## Usage

1. Update `NAIP_FOLDERS` and `FIRE_YEAR_BY_FOLDER` in Cell 3.
2. Optional: set exact fire dates in `FIRE_DATE_BY_FOLDER` (YYYY-MM-DD) for best accuracy.
3. If not using exact dates, set `SAME_YEAR_IS_PREFIRE` in Cell 3.
4. Adjust safety thresholds in Cell 3 (`MAX_MOVE_FILES`, `MAX_UNZIP_FILES`, `MAX_ESTIMATED_UNZIP_GB`).
5. Run all cells once with `DRY_RUN = True` to preview plans and storage impact.
6. Set `DRY_RUN = False` and run the move/unzip cells again to apply changes.

The move and unzip cells perform preflight sanity checks and print counts plus estimated storage expansion before major operations. Before any extraction, each ZIP is tested with `unzip -t`. Archives that fail validation are moved to `_invalid_zips_for_purge/` inside the same NAIP folder, preserving their relative `prefire` or `postfire` path.